In [ ]:
import numpy as np
import plotly.graph_objects as go
from numpy.typing import ArrayLike
from math import pi

font_dict = dict(family="Computer Modern", size=18, color="#7f7f7f")

# Binary variables

In [128]:
class BinaryPairwiseBettingMartingale():

    def __init__(self):
        self.martingale = [1]
        self.observed = {0: 0, 1: 0 }
        self.transitions = {0: 0, 1: 0}
        self.last_observed = None

    @property 
    def parameters(self):
        if sum(list(self.observed.values())) < 1 :
            return None

        params = {}
        for key in [0, 1]:
            if self.observed[key] > 0:
                params[key] = self.transitions[key] / self.observed[key]
            else:
                params[key] = 0
        return params

    @staticmethod
    def validate_sequence_shape(X: ArrayLike):
        if not (len(X.shape) == 1):
            raise ValueError("Sequence must be a one dimensionnal iterable")

    @staticmethod
    def different_values(X1, X2):
        return X1 != X2

    def update_parameters(self, X1, X2):
        self.observed[X1] += 1
        if self.different_values(X1, X2):
            self.transitions[X1] += 1
        return self
    
    def null_hypothesis_likelihood(self, X1, X2):
        return 1/2

    def alternative_hypothesis_likelihood(self, X1, X2):
        params = self.parameters
        diff_last_X1 = self.different_values(self.last_observed, X1)
        diff_last_X2 = self.different_values(self.last_observed, X2)
        
        num = diff_last_X1 * params[self.last_observed] + (1 - diff_last_X1) * (1 - params[self.last_observed])
        num *= params[X1]
        denum = diff_last_X2 * params[self.last_observed] + (1 - diff_last_X2) * (1 - params[self.last_observed])
        denum *= params[X2]
        denum += num

        if denum == 0:
            return 1

        return num / denum


    def bet(self, X1, X2):
        if self.different_values(X1, X2) and len(self.martingale) > 1:
            return ( 
                self.alternative_hypothesis_likelihood(X1, X2) /
                self.null_hypothesis_likelihood(X1, X2)
            )
        return 1

    def run_martingale(self, X:ArrayLike):
        self.validate_sequence_shape(X)

        for i in np.arange(1, len(X) - 1, step=2):
            X2 = X[i]
            X1 = X[i-1]
            m = 1
            if self.last_observed is not None :
                m = self.martingale[-1] * self.bet(X1, X2)
                self.update_parameters(self.last_observed, X1)
            self.martingale.append(m)
            self.update_parameters(X1, X2)
            self.last_observed = X2
        return self

In [129]:
test_Bern = np.random.binomial(1, .7, 3000)
M_Bern = BinaryPairwiseBettingMartingale()
M_Bern = M_Bern.run_martingale(test_Bern)
res_Bern = np.log(M_Bern.martingale)
print(res_Bern.min(), res_Bern.max())


-1.304380712911426 0.6931471805599453


In [130]:
test_Markov = [1]
for i in range(2999):
    current = test_Markov[i]
    if current == 1 :
        test_Markov.append(np.random.binomial(1, .2, 1)[0])
    else:
        test_Markov.append(np.random.binomial(1, .7, 1)[0])
test_Markov = np.array(test_Markov)

M_Markov = BinaryPairwiseBettingMartingale()
M_Markov = M_Markov.run_martingale(test_Markov)
res = np.log(M_Markov.martingale)
print(res.min(), res.max())

0.0 134.0419012684437


In [216]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(x=list(range(len(test_Markov))), y=test_Markov, name='Order 1 Markov Model', mode='markers', opacity=0.5)
)
fig.add_trace(
    go.Scatter(x=list(range(len(test_Bern))), y=test_Bern, name='Bernoulli Sequence', mode='markers', opacity=0.5)
)
fig.update_layout(
    title= f"Test Sequences",
    xaxis_title="Index",
    yaxis_title="Value",
    showlegend=True,
    plot_bgcolor="white",  
    width=800,
    height=600,
    font=font_dict,
    hovermode="y",
    legend=dict(
        x=1,
        y=1,
        xanchor="left",
        yanchor="top",
    )
)
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(x=list(range(len(res))), y=res, name='Order 1 Markov Model')
)
fig.add_trace(
    go.Scatter(x=list(range(len(res_Bern))), y=res_Bern, name='Bernoulli Sequence')
)
fig.update_layout(
    title= f"Likelihood ratio martingales",
    xaxis_title="Index",
    yaxis_title="Value",
    showlegend=True,
    plot_bgcolor="white",  
    width=800,
    height=600,
    font=font_dict,
    hovermode="y",
    legend=dict(
        x=1,
        y=1,
        xanchor="left",
        yanchor="top",
    )
)
fig.show()

# Continuous variables

In [171]:
class ContinuousPairwiseBettingMartingale():

    def __init__(self):
        self.martingale = [1]
        self.num_coeff = 0
        self.denum_coeff = 0
        self.sum_sigma = 0
        self.last_observed = None

    @property 
    def parameters(self):
        return {
            'coeff': self.num_coeff / self.denum_coeff,
            'var' : 1/ (len(self.martingale) * 2 - 2) * self.sum_sigma
        }
        return params

    @staticmethod
    def validate_sequence_shape(X: ArrayLike):
        if not (len(X.shape) == 1):
            raise ValueError("Sequence must be a one dimensionnal iterable")

    def update_parameters(self, X1, X2):
        self.num_coeff += X1 * X2
        self.denum_coeff += X2**2
        self.sum_sigma += (X2 - self.num_coeff / self.denum_coeff * X1) ** 2
        return self
    
    def null_hypothesis_likelihood(self, X1, X2):
        return 1/2

    def alternative_hypothesis_likelihood(self, X1, X2):
        params = self.parameters
        var = params['var']
        coeff = params['coeff']
        
        num = 1/(2 * pi * var)
        expo1 = -1 / (2 * var) * (X1 - coeff * self.last_observed) ** 2
        expo2 = -1 / (2 * var) * (X2 - coeff * X1) ** 2
        num *= np.exp( expo1 + expo2 )
        denum = 1/(2 * pi * var)
        expo1 = -1 / (2 * var) * (X2 - coeff * self.last_observed) ** 2
        expo2 = -1 / (2 * var) * (X1 - coeff * X2) ** 2
        denum *= np.exp( expo1 + expo2 )
        denum += num

        return num / denum

    def bet(self, X1, X2):
        if len(self.martingale) > 1:
            return ( 
                self.alternative_hypothesis_likelihood(X1, X2) /
                self.null_hypothesis_likelihood(X1, X2)
            )
        return 1

    def run_martingale(self, X:ArrayLike):
        self.validate_sequence_shape(X)

        for i in np.arange(1, len(X) - 1, step=2):
            X2 = X[i]
            X1 = X[i-1]
            m = 1
            if self.last_observed is not None :
                m = self.martingale[-1] * self.bet(X1, X2)
                self.update_parameters(self.last_observed, X1)
            self.martingale.append(m)
            self.update_parameters(X1, X2)
            self.last_observed = X2

        return self

In [211]:
test_gaussian = np.random.normal(2, 2, 3000)
M_gaussian = ContinuousPairwiseBettingMartingale()
M_gaussian = M_gaussian.run_martingale(test_gaussian)
res_gaussian = np.log(M_gaussian.martingale)
print(res_gaussian.min(), res_gaussian.max())

-61.59055254102388 1.794809178702795


In [213]:
test_random_walk = np.random.normal(0, 2, 3000).cumsum()
M_rw = ContinuousPairwiseBettingMartingale()
M_rw = M_rw.run_martingale(test_random_walk)
res_rw = np.log(M_rw.martingale)
print(res_rw.min(), res_rw.max())

0.0 130.47638289098919


In [184]:
coeff = .7
test_AR1 = np.random.normal(2, 2, 3000)
for i in np.arange(1, len(test_AR1) - 1, step=1):
    test_AR1[i] = test_AR1[i-1] * coeff + test_AR1[i]
M_ar = ContinuousPairwiseBettingMartingale()
M_ar = M_ar.run_martingale(test_AR1)
res_ar = np.log(M_ar.martingale)
print(M_ar.parameters, res_ar.min(), res_ar.max())


{'coeff': 0.9564407712569435, 'var': 4.5478949605308285} -1.3378003868724662 86.70449829978214


In [215]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(x=list(range(len(test_gaussian))), y=test_gaussian, name='Independent Gaussian')
)
fig.add_trace(
    go.Scatter(x=list(range(len(test_random_walk))), y=test_random_walk, name='Random Walk')
)
fig.add_trace(
    go.Scatter(x=list(range(len(test_AR1))), y=test_AR1, name='AR(1)')
)
fig.update_layout(
    title= f"Test Sequences",
    xaxis_title="Index",
    yaxis_title="Value",
    showlegend=True,
    plot_bgcolor="white",  
    width=800,
    height=600,
    font=font_dict,
    hovermode="y",
    legend=dict(
        x=1,
        y=1,
        xanchor="left",
        yanchor="top",
    )
)
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(x=list(range(len(res_gaussian))), y=res_gaussian, name='Independent Gaussian')
)
fig.add_trace(
    go.Scatter(x=list(range(len(res_rw))), y=res_rw, name='Random Walk')
)
fig.add_trace(
    go.Scatter(x=list(range(len(res_ar))), y=res_ar, name='AR(1)')
)
fig.update_layout(
    title= f"Likelihood ratio martingale sequences",
    xaxis_title="Index",
    yaxis_title="Value",
    showlegend=True,
    plot_bgcolor="white",  
    width=800,
    height=600,
    font=font_dict,
    hovermode="y",
    legend=dict(
        x=1,
        y=1,
        xanchor="left",
        yanchor="top",
    )
)
fig.show()